# TP5 Deep Learning

In [1]:
import torch
import torch.nn as nn

## 1 Toy Dataset

In [2]:
def generate_disc_set(nb: int = 1000) -> torch.Tensor:
    """
    Generate a toy dataset
    """
    input = torch.empty(nb, 2).uniform_(-1, 1)
    assert input.min()  >= -1.0 and input.max() <= 1.0, f"Input values should be in the range [-1, 1] ! Current range: [{input.min()}, {input.max()}]"

    output_class = torch.zeros(nb)
    output_class[input[:, 0]**2 + input[:, 1]**2 < 2 / torch.pi] = 1.0

    output = torch.zeros(nb, 2, dtype = torch.float64)
    output[:, 0] = output_class != 1
    output[:, 1] = output_class == 1
    return input, output

In [3]:
train_input, train_target = generate_disc_set(1000)
train_input.sub_(train_input.mean(0)).div_(train_input.std(0))

test_input, test_target = generate_disc_set(1000)
test_input.sub_(test_input.mean(0)).div_(test_input.std(0))



tensor([[-0.9612, -1.4648],
        [-1.0946, -0.7892],
        [-0.3188, -0.0339],
        ...,
        [-0.5756, -1.7772],
        [-1.3387, -1.5393],
        [-1.4784,  0.5882]])

In [4]:
def train_model(model: torch.nn.Module, train_input: torch.Tensor, train_target: torch.Tensor, nb_epochs: int = 250, mini_batch_size: int = 100, eta: float = 0.1):
    criterion = nn.CrossEntropyLoss()
    for epoch in range(nb_epochs):
        for input, target in zip(train_input.split(mini_batch_size), train_target.split(mini_batch_size)):
            output = model(input)
            loss = criterion(output, target)
            model.zero_grad()
            loss.backward()
            with torch.no_grad():
                for p in model.parameters():
                    p -= eta * p.grad
        

def train_model_v2(model: torch.nn.Module, train_input: torch.Tensor, train_target: torch.Tensor, nb_epochs: int = 250, mini_batch_size: int = 100, eta: float = 0.1):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.sgd(model.parameters(), lr = eta)
    for epoch in range(nb_epochs):
        for input, target in zip(train_input.split(mini_batch_size), train_target.split(mini_batch_size)):
            output = model(input)
            loss = criterion(output, target)
            optimizer.zero_grad()
            model.backward()
            optimizer.step()


def compute_nb_errors(model: torch.nn.Model, data_input: torch.Tensor, data_target: torch.Tensor, mini_batch_size: int = 100) -> int:
    nb_errors = 0
    for b in range(0, data_input.size(0), mini_batch_size):
        output = model(data_input.narrow(0, b, mini_batch_size))
        pred_idx = output.max(1)[1]
        target_idx = data_target.narrow(0, b, mini_batch_size).max(1)[1]
        nb_errors += (pred_idx != target_idx).sum().item()
    return nb_errors

In [5]:
def create_shallow_model():
    return nn.Sequential(nn.Linear(2, 128), nn.ReLU(), nn.Linear(128, 2), nn.Sigmoid())

def create_deep_model():
    model = nn.Sequential()
    for i in range(2, 8):
        model.append(nn.Linear(2**(i - 1), 2**i))
        model.append(nn.ReLU())
    model.append(nn.Linear(2**7, 2))
    model.append(nn.Sigmoid())
    return model

In [6]:
model1 = create_shallow_model()
model2 = create_deep_model()

In [11]:
for std in [-1, 1e-3, 1e-2, 1e-1, 1.0, 1e1]:
    for m in [create_shallow_model, create_deep_model]:
        model = m()
        if std > 0:
            with torch.no_grad():
                for p in model.parameters():
                    p.data.normal_(0, std)
        train_model(model, train_input, train_target)
        print(f"Model: {m.__name__}, std: {std}, train error rate: {100 * compute_nb_errors(model, train_input, train_target) / train_input.size(0):.2f}% test error rate: {100 * compute_nb_errors(model, test_input, test_target) / test_input.size(0):.2f}%")

Model: create_shallow_model, std: -1, train error rate: 1.60% test error rate: 3.50%
Model: create_deep_model, std: -1, train error rate: 3.80% test error rate: 4.70%
Model: create_shallow_model, std: 0.001, train error rate: 21.10% test error rate: 22.00%
Model: create_deep_model, std: 0.001, train error rate: 49.20% test error rate: 50.20%
Model: create_shallow_model, std: 0.01, train error rate: 6.00% test error rate: 6.40%
Model: create_deep_model, std: 0.01, train error rate: 49.20% test error rate: 50.20%
Model: create_shallow_model, std: 0.1, train error rate: 3.40% test error rate: 4.60%
Model: create_deep_model, std: 0.1, train error rate: 49.20% test error rate: 50.20%
Model: create_shallow_model, std: 1.0, train error rate: 52.90% test error rate: 52.70%
Model: create_deep_model, std: 1.0, train error rate: 50.80% test error rate: 49.80%
Model: create_shallow_model, std: 10.0, train error rate: 59.60% test error rate: 58.10%
Model: create_deep_model, std: 10.0, train error r